In [25]:
import joblib
import numpy as np
import pandas as pd
import shap

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, confusion_matrix

In [2]:
df_churn = pd.read_excel('C:/code/github/churn-project/Telco Churn Solutions/Telco_customer_churn_adapted_v2.xlsx')
churn_model = joblib.load('C:/code/github/churn-project/Telco Churn Solutions/churn_model.joblib')

In [3]:
# preprocessing
df_churn['Internet'] = df_churn['Video Product'].apply(lambda x: 0 if x == 'No internet access' else 1)

In [4]:
# model prediction
X = df_churn[['Tenure Months','Monthly Purchase (Thou. IDR)', 'Device Class', 'Call Center', 'Use MyApp', 'Payment Method', 'Internet',
        'Games Product', 'Music Product', 'Education Product', 'Video Product']]
y = df_churn['Churn Label'].apply(lambda x: 1 if x == 'Yes' else 0)

In [5]:
predictions = churn_model.predict(X)
print(accuracy_score(y, predictions))

0.8067584836007383


In [6]:
churn_model.predict_proba(X.head(1))[:, 1]

array([0.23575619], dtype=float32)

In [7]:
X.head(1)

,Tenure Months,Monthly Purchase (Thou. IDR),Device Class,Call Center,Use MyApp,Payment Method,Internet,Games Product,Music Product,Education Product,Video Product
0,2,70.005,Mid End,No,No,Digital Wallet,1,Yes,Yes,No,No


In [8]:
new_data = pd.DataFrame([{
    'Tenure Months': '2',
    'Monthly Purchase (Thou. IDR)': '2000',
    'Device Class': 'Low End',
    'Call Center': 1, 
    'Use MyApp': 1,
    'Payment Method': 'Pulsa', 
    'Internet': 1,
    'Games Product': 0,
    'Music Product': 1,
    'Education Product': 1,
    'Video Product': 0
}])

In [9]:
new_data

,Tenure Months,Monthly Purchase (Thou. IDR),Device Class,Call Center,Use MyApp,Payment Method,Internet,Games Product,Music Product,Education Product,Video Product
0,2,2000,Low End,1,1,Pulsa,1,0,1,1,0


In [10]:
churn_model.predict_proba(new_data)[:, 1]

array([0.5445823], dtype=float32)

In [ ]:
# SHAP

In [11]:
churn_model

Pipeline(steps=[('encoder',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('encoder',
                                                  TargetEncoder(handle_unknown='ignore'),
                                                  ['Device Class',
                                                   'Call Center', 'Use MyApp',
                                                   'Payment Method', 'Internet',
                                                   'Games Product',
                                                   'Music Product',
                                                   'Education Product',
                                                   'Video Product'])])),
                ('scaler', StandardScaler()),
                ('model',
                 XGBClassifier(base_score=None, booster=None, ca...
                               feature_types=None, gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=3, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=100, n_jobs=None,
                               num_parallel_tree=None, random_state=42, ...))])

In [17]:
X = df_churn[['Tenure Months','Monthly Purchase (Thou. IDR)', 'Device Class', 'Call Center', 'Use MyApp', 'Payment Method', 
              'Internet', 'Games Product', 'Music Product', 'Education Product', 'Video Product']]
xgb_classifier = churn_model.named_steps['model']
encoder = churn_model.named_steps['encoder']
scaler = churn_model.named_steps['scaler']

In [18]:
transformed_data = encoder.transform(X)
transformed_data = scaler.transform(transformed_data)
preprocessed_df = pd.DataFrame(transformed_data)
transformed_columns = encoder.get_feature_names_out(input_features=X.columns).tolist()

In [23]:
explainer = shap.Explainer(xgb_classifier)
shap_values = explainer.shap_values(preprocessed_df)
shap_values_df = pd.DataFrame(shap_values, columns=transformed_columns)

In [62]:
def compile_shap_values(row):
    shap_texts = []
    for col_name, value in zip(transformed_columns, row):
        shap_texts.append(f"{col_name}: {value:.4f}")
    return ', '.join(shap_texts)

feature_name_mapping = {
    'encoder__Device Class': "User's Device Type",
    'encoder__Payment Method':"User's Payment Method",
    'encoder__Call Center': 'Call Center usage',
    'encoder__Use MyApp': 'MyApp usage',
    'encoder__Internet':"User's Internet access",
    'encoder__Games Product':'Games Product usage',
    'encoder__Music Product':'Music Product usage',
    'encoder__Education Product':'Education Product usage',
    'encoder__Video Product':'Video Product usage',
    'remainder__Tenure Months':"User's Tenure Months",
    'remainder__Monthly Purchase (Thou. IDR)':"User's Monthly Purchase"
}

def get_most_impactful_features(row):
    top_indices = np.argsort(row)[-2:]

    feature_names = [feature_name_mapping.get(shap_values_df.columns[idx], shap_values_df.columns[idx]) for idx in top_indices]

    sorted_features = sorted(feature_names, reverse=True)  
    return sorted_features[0] if sorted_features[0] == sorted_features[1] else ' and '.join(sorted_features)

transformed_columns = [feature_name_mapping.get(name, name) for name in transformed_columns]

df_churn['shap'] = shap_values_df.apply(compile_shap_values, axis=1)
df_churn['rules_desc'] = shap_values_df.apply(get_most_impactful_features, axis=1)
df_churn['rules_desc'] = df_churn['rules_desc'].apply(lambda x: f"{x} are the top two most contributing features for this specific customer")

In [63]:
df_churn['rules_desc'][0]

"User's Tenure Months and Call Center usage are the top two most contributing features for this specific customer"